# LPIPS


Learned Perceptual Image Patch Similarity. Mide diferencia perceptual usando características extraídas de redes neuronales (VGG/AlexNet). Valores más bajos = más similares. Alta correlación con percepción humana.

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io
from skimage.util import img_as_ubyte
import lpips

# ---------------------------
# CONFIGURACIÓN
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_lpips = lpips.LPIPS(net='alex').to(device)  # red elegida: AlexNet (mejor correlación en BAPPS, Zhang et al., 2018)

# ---------------------------
# VALIDACIÓN DE FORMATO (sin conversión)
# ---------------------------

def es_rgb_valida(img):
    """
    Verifica si una imagen ya está en formato RGB de 3 canales,
    sin realizar ninguna conversión.

    Retorna
    -------
    bool
        True si la imagen es RGB (3 canales), False en cualquier
        otro caso (escala de grises, RGBA, u otro formato).
    """
    return img.ndim == 3 and img.shape[-1] == 3

def load_tensor(path, device):
    """
    Carga una imagen y la convierte a tensor solo si ya está en
    formato RGB. Si no lo está (escala de grises, RGBA, etc.),
    retorna None para que sea omitida.

    Retorna
    -------
    torch.Tensor o None
        Tensor listo para LPIPS, o None si la imagen debe omitirse.
    """
    img = io.imread(path)

    if not es_rgb_valida(img):
        return None

    img = img_as_ubyte(img)
    img = img.astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# LPIPS
# ---------------------------

def calcular_lpips(img_orig_path, img_comp_path):
    """
    Calcula la distancia perceptual LPIPS entre dos imágenes.

    Si alguna de las dos imágenes no está en formato RGB (por
    ejemplo, escala de grises o RGBA), la comparación se omite
    y se retorna None en lugar de forzar una conversión.

    Parámetros
    ----------
    img_orig_path : str
        Ruta de la imagen original.
    img_comp_path : str
        Ruta de la imagen comprimida o transformada.

    Retorna
    -------
    float o None
        Distancia perceptual LPIPS, o None si alguna imagen
        fue omitida por no ser RGB.
    """
    orig = load_tensor(img_orig_path, device)
    comp = load_tensor(img_comp_path, device)

    if orig is None or comp is None:
        return None

    orig, comp = match_size(orig, comp)

    with torch.no_grad():
        dist = loss_fn_lpips(orig, comp)

    return dist.item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\lpips\weights\v0.1\alex.pth
